# Stage-1 Candidate Selector Training (Google Colab)

This notebook trains the Stage-1 candidate selector pipeline:

1. Bi-encoder retrieval model  
2. Cross-encoder reranker

It uses the expanded training data (`stage1_training_input_sample.jsonl`) and supports early stopping with per-epoch validation metrics.

In [ ]:
# @title 1) Install dependencies
!pip -q install -U sentence-transformers

import sys
print("Python:", sys.version)

import torch
import numpy as np
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# @title 2) Clone repo and checkout branch
import os

REPO_URL = "https://github.com/Shaurya11gupta/DataMigrationCrane.git"  # @param {type:"string"}
BRANCH = "cursor/string-transformation-data-268f"  # @param {type:"string"}
WORKDIR = "/content/DataMigrationCrane"  # @param {type:"string"}

if not os.path.exists(WORKDIR):
    !git clone "{REPO_URL}" "{WORKDIR}"

%cd {WORKDIR}
!git fetch origin "{BRANCH}"
!git checkout "{BRANCH}"
!git pull origin "{BRANCH}"
!git status -sb

In [ ]:
# @title 3) (Optional) Regenerate Stage-1 training data
REGENERATE_DATA = False  # @param {type:"boolean"}

if REGENERATE_DATA:
    !python3 expand_stage1_training_data.py

!python3 - <<'PY'
import json
from pathlib import Path
p = Path('stage1_training_input_sample.jsonl')
lines = [x for x in p.read_text(encoding='utf-8').splitlines() if x.strip()]
print('rows:', len(lines))
first = json.loads(lines[0])
print('sample query_id:', first.get('query_id'))
print('sample split:', first.get('split'))
PY

In [ ]:
# @title 4) Train Stage-1 models (with early stopping)
import subprocess

OUTPUT_DIR = "artifacts/stage1_candidate_selector"  # @param {type:"string"}
BI_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # @param {type:"string"}
CROSS_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # @param {type:"string"}

# You can set large max epochs safely now; early stopping will stop when needed.
BI_EPOCHS = 50  # @param {type:"integer"}
CROSS_EPOCHS = 30  # @param {type:"integer"}

BI_BATCH_SIZE = 32  # @param {type:"integer"}
CROSS_BATCH_SIZE = 16  # @param {type:"integer"}

BI_LR = 2e-5  # @param {type:"number"}
CROSS_LR = 1e-5  # @param {type:"number"}

BI_MONITOR_K = 10  # @param {type:"integer"}
CROSS_MONITOR_K = 10  # @param {type:"integer"}

BI_PATIENCE = 4  # @param {type:"integer"}
CROSS_PATIENCE = 3  # @param {type:"integer"}
BI_MIN_DELTA = 0.0005  # @param {type:"number"}
CROSS_MIN_DELTA = 0.0005  # @param {type:"number"}
BI_MIN_EPOCHS = 2  # @param {type:"integer"}
CROSS_MIN_EPOCHS = 2  # @param {type:"integer"}

DISABLE_EARLY_STOPPING = False  # @param {type:"boolean"}

cmd = [
    "python3", "candidate_selector_stage1.py", "train",
    "--input-jsonl", "stage1_training_input_sample.jsonl",
    "--output-dir", OUTPUT_DIR,
    "--bi-model-name", BI_MODEL_NAME,
    "--cross-model-name", CROSS_MODEL_NAME,
    "--bi-epochs", str(BI_EPOCHS),
    "--cross-epochs", str(CROSS_EPOCHS),
    "--bi-batch-size", str(BI_BATCH_SIZE),
    "--cross-batch-size", str(CROSS_BATCH_SIZE),
    "--bi-lr", str(BI_LR),
    "--cross-lr", str(CROSS_LR),
    "--bi-monitor-k", str(BI_MONITOR_K),
    "--cross-monitor-k", str(CROSS_MONITOR_K),
    "--bi-patience", str(BI_PATIENCE),
    "--cross-patience", str(CROSS_PATIENCE),
    "--bi-min-delta", str(BI_MIN_DELTA),
    "--cross-min-delta", str(CROSS_MIN_DELTA),
    "--bi-min-epochs", str(BI_MIN_EPOCHS),
    "--cross-min-epochs", str(CROSS_MIN_EPOCHS),
]
if DISABLE_EARLY_STOPPING:
    cmd.append("--disable-early-stopping")

print("Running:\n", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# @title 5) Inspect training metadata and best checkpoints
import json
from pathlib import Path

meta_path = Path(OUTPUT_DIR) / "training_metadata.json"
assert meta_path.exists(), f"Metadata not found: {meta_path}"

meta = json.loads(meta_path.read_text(encoding="utf-8"))
print("Bi summary:")
print(json.dumps(meta.get("bi_train_summary", {}), indent=2))
print("\nCross summary:")
print(json.dumps(meta.get("cross_train_summary", {}), indent=2))
print("\nFinal val metrics:")
print(json.dumps(meta.get("metrics", {}).get("pipeline_val", {}), indent=2))

In [ ]:
# @title 6) Plot per-epoch monitor values
import matplotlib.pyplot as plt

def plot_monitor(history, title):
    if not history:
        print(f"No history for {title}")
        return
    epochs = [h["epoch"] for h in history]
    vals = [h["monitor_value"] for h in history]
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, vals, marker="o")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Monitor value")
    plt.grid(True, alpha=0.3)
    plt.show()

plot_monitor(meta.get("bi_epoch_history", []), "Bi-encoder monitor by epoch")
plot_monitor(meta.get("cross_epoch_history", []), "Cross-encoder monitor by epoch")

In [ ]:
# @title 7) Inference demo on one query from dataset
import json
from collections import defaultdict
from candidate_selector_stage1 import CandidateSelectorStage1, load_training_records

records = load_training_records("stage1_training_input_sample.jsonl")
grouped = defaultdict(list)
for r in records:
    grouped[r.query_id].append(r)

query_id = next(iter(grouped.keys()))
recs = grouped[query_id]

target = {
    "table": recs[0].target.table,
    "column": recs[0].target.column,
    "type": recs[0].target.type,
    "description": recs[0].target.description,
}
candidate_sets = []
for r in recs:
    candidate_sets.append(
        {
            "id": r.candidate_set.id,
            "columns": [
                {
                    "table": c.table,
                    "column": c.column,
                    "type": c.type,
                    "description": c.description,
                }
                for c in r.candidate_set.columns
            ],
            "join_path": r.candidate_set.join_path,
            "transform_hint": r.candidate_set.transform_hint,
        }
    )

selector = CandidateSelectorStage1(
    biencoder_path=f"{OUTPUT_DIR}/biencoder",
    cross_encoder_path=f"{OUTPUT_DIR}/cross_encoder",
)
ranked = selector.rank(
    target=recs[0].target,
    candidate_sets=[r.candidate_set for r in recs],
    retrieval_k=30,
    top_k=10,
)

print("Query ID:", query_id)
print(json.dumps(target, indent=2))
print("\nTop ranked candidates:")
print(json.dumps(ranked, indent=2))

## Notes

- If you are testing quickly, start with lower max epochs (for example bi=6, cross=4).  
- For full training, high max epochs are safe because early stopping restores the best checkpoint.  
- Artifacts are saved under `artifacts/stage1_candidate_selector/`.